## Findings

- Refunds after the first day total £359,211 across 7,071 cancellation invoices, 1.9% of sales revenue (£19.23M). Orders cancelled within a day were treated separately earlier (£404,143 of sales value).
- Monthly refund rates are typically 1.1% to 2.6%, with no post-Christmas surge. April 2011 (6.2%) is almost entirely one invoice (C550456, about £23,000, 73% of the month's refunds); without it the month is 1.7%. The same customer placed an identical order 12 minutes after the cancellation, which suggests the order was re-entered rather than returned, although the data cannot confirm this. May 2010 (3.6%) is a broad rise (817 refund lines across 369 invoices) with no single cause visible in this data.
- Spain (13.3%), Denmark (6.0%) and France (5.3%) appear to have high refund rates, but each is driven by one customer. Without that customer the rates are 0.7%, 1.3% and 1.2%, below the UK's 1.8%.
- Lapsing regulars have the highest segment refund rate (3.2%, against 1.9% for the core segment), but about a quarter of that segment's refunds is the single April invoice above. Without it the rate is about 2.3%, a small difference.

## Limitations
- Refunds are compared with sales in the same period, not with the specific sales being returned, so monthly rates are approximate.
- Reversals can only be attributed to customers when a customer ID exists (about 2% of refund value has none).
- Small countries have few customers, so rates there reflect individual accounts.

In [1]:
import pandas as pd

sales = pd.read_csv(
    "../data/sales_final.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Invoice": str, "StockCode": str},
)
canc = pd.read_csv(
    "../data/cancellations_final.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Invoice": str, "StockCode": str},
)
canc["Refund"] = -canc["Revenue"]

print("Cancellation invoices:", canc["Invoice"].nunique())
print("Cancellation lines:", len(canc))
print("Total refunded:", round(canc["Refund"].sum(), 2))
print("Sales revenue:", round(sales["Revenue"].sum(), 2))
print("Refunds as % of sales:", round(100 * canc["Refund"].sum() / sales["Revenue"].sum(), 1), "%")
print("Cancellation rows with non-negative quantity:", (canc["Quantity"] >= 0).sum())

Cancellation invoices: 7071
Cancellation lines: 16563
Total refunded: 359210.88
Sales revenue: 19234419.48
Refunds as % of sales: 1.9 %
Cancellation rows with non-negative quantity: 0


In [2]:
sales["month"] = sales["InvoiceDate"].dt.to_period("M")
canc["month"] = canc["InvoiceDate"].dt.to_period("M")

monthly = pd.DataFrame({
    "sales": sales.groupby("month")["Revenue"].sum(),
    "refunds": canc.groupby("month")["Refund"].sum(),
}).fillna(0)
monthly["refund_pct"] = (100 * monthly["refunds"] / monthly["sales"]).round(1)
monthly.round(0)

,sales,refunds,refund_pct
month,,,
2009-12,794856.0,15314.0,2.0
2010-01,611939.0,7136.0,1.0
2010-02,536778.0,11677.0,2.0
2010-03,758871.0,8349.0,1.0
2010-04,643257.0,8088.0,1.0
2010-05,627700.0,22664.0,4.0
2010-06,682451.0,12693.0,2.0
2010-07,624267.0,11260.0,2.0
2010-08,671417.0,9778.0,2.0


In [3]:
top = sales.groupby("Country")["Revenue"].sum().nlargest(10).index
by_country = pd.DataFrame({
    "sales": sales.groupby("Country")["Revenue"].sum(),
    "refunds": canc.groupby("Country")["Refund"].sum(),
}).loc[top].fillna(0)
by_country["refund_pct"] = (100 * by_country["refunds"] / by_country["sales"]).round(1)
by_country.round(0).sort_values("refund_pct", ascending=False)

,sales,refunds,refund_pct
Country,,,
Spain,97767.0,12955.0,13.0
Denmark,67423.0,4067.0,6.0
France,309853.0,16422.0,5.0
United Kingdom,16404872.0,288172.0,2.0
EIRE,614229.0,11020.0,2.0
Germany,383169.0,8596.0,2.0
Sweden,86081.0,1736.0,2.0
Netherlands,549773.0,3678.0,1.0
Australia,167423.0,1216.0,1.0


In [4]:
by_cust = canc.dropna(subset=["Customer ID"]).groupby("Customer ID")["Refund"].sum().sort_values(ascending=False)
print("Customers with refunds:", len(by_cust))
print("Top 10 customers' share of identified refunds:", round(100 * by_cust.iloc[:10].sum() / by_cust.sum(), 1), "%")
print("Refund value with no customer ID:", round(canc.loc[canc["Customer ID"].isna(), "Refund"].sum(), 2))

Customers with refunds: 2354
Top 10 customers' share of identified refunds: 25.4 %
Refund value with no customer ID: 6472.54


In [5]:
for mth in ["2011-04", "2010-05"]:
    g = canc[canc["month"] == pd.Period(mth)]
    print(mth, "| refund lines:", len(g), "| invoices:", g["Invoice"].nunique(),
          "| top 5 lines' share of month:", round(100 * g["Refund"].nlargest(5).sum() / g["Refund"].sum(), 1), "%")
    display(g.nlargest(5, "Refund")[["Invoice", "InvoiceDate", "Description", "Quantity", "Price", "Refund", "Customer ID", "Country"]])

2011-04 | refund lines: 479 | invoices: 208 | top 5 lines' share of month: 72.7 %


,Invoice,InvoiceDate,Description,Quantity,Price,Refund,Customer ID,Country
11101,C550456,2011-04-18 13:08:00,FAIRY CAKE FLANNEL ASSORTED COLOUR,-3114,2.10,6539.4,15749.0,United Kingdom
11099,C550456,2011-04-18 13:08:00,WHITE HANGING HEART T-LIGHT HOLDER,-1930,2.55,4921.5,15749.0,United Kingdom
11097,C550456,2011-04-18 13:08:00,DOORMAT FAIRY CAKE,-670,6.75,4522.5,15749.0,United Kingdom
11100,C550456,2011-04-18 13:08:00,GIN + TONIC DIET METAL SIGN,-2000,1.85,3700.0,15749.0,United Kingdom
11098,C550456,2011-04-18 13:08:00,TEA TIME PARTY BUNTING,-1300,2.55,3315.0,15749.0,United Kingdom


2010-05 | refund lines: 817 | invoices: 369 | top 5 lines' share of month: 15.2 %


,Invoice,InvoiceDate,Description,Quantity,Price,Refund,Customer ID,Country
3710,C508455,2010-05-14 16:38:00,WILLOW BRANCH LIGHTS.,-312,5.45,1700.4,16077.0,United Kingdom
4035,C510214,2010-05-28 08:16:00,RED RETROSPOT CAKE STAND,-60,9.95,597.0,15061.0,United Kingdom
4020,C510211,2010-05-28 08:06:00,TEA TIME PARTY BUNTING,-144,3.75,540.0,14147.0,United Kingdom
3385,C507015,2010-05-05 14:28:00,SET OF 12 ROSE BOTANICAL T-LIGHTS,-120,2.55,306.0,15097.0,United Kingdom
4078,C510252,2010-05-28 10:37:00,PACK OF 6 PANNETONE GIFT BOXES,-144,2.10,302.4,15061.0,United Kingdom


In [6]:
check = canc[canc["Country"].isin(["Spain", "Denmark", "France"])].dropna(subset=["Customer ID"])
rows = []
for c, g in check.groupby("Country"):
    r = g.groupby("Customer ID")["Refund"].sum().sort_values(ascending=False)
    rows.append({
        "country": c,
        "customers_with_refunds": len(r),
        "refund_lines": len(g),
        "top_customer_share_pct": round(100 * r.iloc[0] / r.sum(), 1),
    })
pd.DataFrame(rows)

,country,customers_with_refunds,refund_lines,top_customer_share_pct
0,Denmark,5,17,89.7
1,France,40,308,78.1
2,Spain,14,81,95.4


In [8]:
seg = pd.read_csv("../data/rfm_segments.csv").set_index("Customer ID")

s_sales = sales.dropna(subset=["Customer ID"]).astype({"Customer ID": int}).groupby("Customer ID")["Revenue"].sum()
s_ref = canc.dropna(subset=["Customer ID"]).astype({"Customer ID": int}).groupby("Customer ID")["Refund"].sum()

seg["sales"] = s_sales
seg["refunds"] = s_ref
seg["refunds"] = seg["refunds"].fillna(0)

t = seg.groupby("segment_name")[["sales", "refunds"]].sum()
t["refund_pct"] = (100 * t["refunds"] / t["sales"]).round(1)
t["share_of_refunds_pct"] = (100 * t["refunds"] / t["refunds"].sum()).round(1)
t.round(0)

,sales,refunds,refund_pct,share_of_refunds_pct
segment_name,,,,
Core high-value,12234254.0,232684.0,2.0,66.0
Lapsing regulars,2758285.0,87805.0,3.0,25.0
Lost one-time buyers,624595.0,13613.0,2.0,4.0
Newer customers,1046641.0,17230.0,2.0,5.0


In [9]:

monthly.round({"sales": 0, "refunds": 0, "refund_pct": 1})

,sales,refunds,refund_pct
month,,,
2009-12,794856.0,15314.0,1.9
2010-01,611939.0,7136.0,1.2
2010-02,536778.0,11677.0,2.2
2010-03,758871.0,8349.0,1.1
2010-04,643257.0,8088.0,1.3
2010-05,627700.0,22664.0,3.6
2010-06,682451.0,12693.0,1.9
2010-07,624267.0,11260.0,1.8
2010-08,671417.0,9778.0,1.5


In [10]:
for mth in ["2011-04", "2010-05"]:
    g = canc[canc["month"] == pd.Period(mth)]
    print(mth, "| refund lines:", len(g), "| invoices:", g["Invoice"].nunique(),
          "| top 5 lines' share of month:", round(100 * g["Refund"].nlargest(5).sum() / g["Refund"].sum(), 1), "%")

apr = canc[canc["month"] == pd.Period("2011-04")]
print("April 2011 refunds excluding invoice C550456:", round(apr.loc[apr["Invoice"] != "C550456", "Refund"].sum(), 2))

2011-04 | refund lines: 479 | invoices: 208 | top 5 lines' share of month: 72.7 %
2010-05 | refund lines: 817 | invoices: 369 | top 5 lines' share of month: 15.2 %
April 2011 refunds excluding invoice C550456: 8615.13


In [11]:
rows = []
for c in ["Spain", "Denmark", "France"]:
    s = sales[sales["Country"] == c]
    r = canc[canc["Country"] == c]
    top_id = r.dropna(subset=["Customer ID"]).groupby("Customer ID")["Refund"].sum().idxmax()
    s2 = s[s["Customer ID"] != top_id]
    r2 = r[r["Customer ID"] != top_id]
    rows.append({
        "country": c,
        "top_customer_id": int(top_id),
        "rate_all_%": round(100 * r["Refund"].sum() / s["Revenue"].sum(), 1),
        "rate_without_top_customer_%": round(100 * r2["Refund"].sum() / s2["Revenue"].sum(), 1),
        "top_customer_sales": round(s.loc[s["Customer ID"] == top_id, "Revenue"].sum()),
        "top_customer_refunds": round(r.loc[r["Customer ID"] == top_id, "Refund"].sum()),
    })
pd.DataFrame(rows)

,country,top_customer_id,rate_all_%,rate_without_top_customer_%,top_customer_sales,top_customer_refunds
0,Spain,12454,13.3,0.7,16460,12361
1,Denmark,13902,6.0,1.3,34023,3648
2,France,14277,5.3,1.2,16789,12829


In [12]:
display(t.round({"sales": 0, "refunds": 0, "refund_pct": 1, "share_of_refunds_pct": 1}))

for name, g in seg.groupby("segment_name"):
    r = g["refunds"].sort_values(ascending=False)
    print(name,
          "| customers with refunds:", int((r > 0).sum()),
          "| top customer share:", round(100 * r.iloc[0] / r.sum(), 1), "%",
          "| top 10 share:", round(100 * r.iloc[:10].sum() / r.sum(), 1), "%")

print("Identified refunds:", round(canc.dropna(subset=["Customer ID"])["Refund"].sum(), 2))
print("Assigned to segments:", round(seg["refunds"].sum(), 2))
print("Customer 15749 segment:", seg.loc[15749, "segment_name"])

,sales,refunds,refund_pct,share_of_refunds_pct
segment_name,,,,
Core high-value,12234254.0,232684.0,1.9,66.2
Lapsing regulars,2758285.0,87805.0,3.2,25.0
Lost one-time buyers,624595.0,13613.0,2.2,3.9
Newer customers,1046641.0,17230.0,1.6,4.9


Core high-value | customers with refunds: 929 | top customer share: 5.5 % | top 10 share: 30.1 %
Lapsing regulars | customers with refunds: 739 | top customer share: 26.2 % | top 10 share: 44.4 %
Lost one-time buyers | customers with refunds: 314 | top customer share: 8.8 % | top 10 share: 26.0 %
Newer customers | customers with refunds: 349 | top customer share: 14.2 % | top 10 share: 42.3 %
Identified refunds: 352738.34
Assigned to segments: 351332.21
Customer 15749 segment: Lapsing regulars


In [13]:
missing = canc.dropna(subset=["Customer ID"]).astype({"Customer ID": int})
missing = missing[~missing["Customer ID"].isin(seg.index)]
print("Refund value from customers not in any segment:", round(missing["Refund"].sum(), 2))
print("Customers:", missing["Customer ID"].nunique())

lap = seg[seg["segment_name"] == "Lapsing regulars"]
inv = canc.loc[canc["Invoice"] == "C550456", "Refund"].sum()
print("Invoice C550456 refund:", round(inv, 2))
print("Lapsing regulars refund rate excluding that invoice:",
      round(100 * (lap["refunds"].sum() - inv) / lap["sales"].sum(), 2), "%")

Refund value from customers not in any segment: 1406.13
Customers: 23
Invoice C550456 refund: 22998.4
Lapsing regulars refund rate excluding that invoice: 2.35 %
